# 02 · Cleaning & QA

Runs QC checks on all raw parquets and writes cleaned files + a QC report.

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
from cleaning import clean_dgstats

## 2.1 DGStats QC

In [2]:
dgstats = pd.read_parquet(RAW / 'dgstats_raw.parquet')
print(f'Raw rows: {len(dgstats):,}')

# Duplicate check
dupes = dgstats.duplicated('application_id').sum()
print(f'Duplicate application_id rows: {dupes}')
dgstats = dgstats.drop_duplicates('application_id')

# Null critical fields
null_size = dgstats['system_size_dc'].isna().sum()
null_date = dgstats['app_approved_date'].isna().sum()
print(f'Null system_size_dc: {null_size} | Null app_approved_date: {null_date}')

# Assert <10% missing on key fields
for col in ['system_size_dc', 'app_approved_date', 'zip_code']:
    miss_pct = dgstats[col].isna().mean()
    assert miss_pct < 0.10, f'{col} missing rate {miss_pct:.1%} exceeds 10% threshold'
    print(f'{col}: {miss_pct:.2%} missing — OK')

Raw rows: 1,503,615
Duplicate application_id rows: 0


Null system_size_dc: 0 | Null app_approved_date: 0
system_size_dc: 0.00% missing — OK
app_approved_date: 0.00% missing — OK
zip_code: 0.00% missing — OK


In [3]:
dgstats_clean = clean_dgstats(dgstats)
print(f'Clean rows: {len(dgstats_clean):,}')
print(f'Size outliers (>100 kW residential): {dgstats_clean["size_outlier"].sum()}')
dgstats_clean.to_parquet(PROC / 'dgstats_clean.parquet', index=False)

Clean rows: 1,500,462
Size outliers (>100 kW residential): 259


## 2.2 CAISO Monthly QC

In [4]:
caiso = pd.read_parquet(RAW / 'caiso_monthly_raw.parquet')
print(f'CAISO monthly rows: {len(caiso):,}  (expected 108)')
print(caiso.dtypes)

# Row count check: 9 years × 12 months
assert len(caiso) == 108, f'Expected 108 rows, got {len(caiso)}'

# Coverage check: 2015–2023 all present
for yr in range(2015, 2024):
    yr_rows = (caiso['year'] == yr).sum()
    status = 'OK' if yr_rows == 12 else f'WARN (expected 12, got {yr_rows})'
    print(f'{yr}: {yr_rows} months — {status}')

# Null checks
for col in ['demand_gwh', 'solar_gwh', 'wind_gwh']:
    nulls = caiso[col].isna().sum()
    print(f'{col} nulls: {nulls}')

# Plausibility: net_load_ratio should be between 0 and 1
ratio_ok = caiso['net_load_ratio'].between(0, 1).all()
print(f'net_load_ratio in [0, 1]: {"PASS" if ratio_ok else "WARN"}')

CAISO monthly rows: 108  (expected 108)
year                      int32
month                     int32
demand_gwh              float64
solar_gwh               float64
wind_gwh                float64
total_generation_gwh    float64
net_load_gwh            float64
net_load_ratio          float64
monthly_ramp_gwh        float64
dtype: object
2015: 12 months — OK
2016: 12 months — OK
2017: 12 months — OK
2018: 12 months — OK
2019: 12 months — OK
2020: 12 months — OK
2021: 12 months — OK
2022: 12 months — OK
2023: 12 months — OK
demand_gwh nulls: 0
solar_gwh nulls: 0
wind_gwh nulls: 0
net_load_ratio in [0, 1]: PASS


In [5]:
# Solar penetration sanity check: solar_gwh / demand_gwh should be reasonable
caiso['solar_pct'] = caiso['solar_gwh'] / caiso['demand_gwh']
high_solar = caiso[caiso['solar_pct'] > 0.5]
if not high_solar.empty:
    print(f'WARNING: {len(high_solar)} months with solar > 50% of demand — inspect these:')
    print(high_solar[['year', 'month', 'solar_pct']])
else:
    print('Solar penetration sanity check: PASS (all months < 50%)')

caiso = caiso.drop(columns=['solar_pct'])
caiso.to_parquet(PROC / 'caiso_clean.parquet', index=False)
print('Saved caiso_clean.parquet')

Solar penetration sanity check: PASS (all months < 50%)
Saved caiso_clean.parquet


## 2.3 QC Report

In [6]:
qc_lines = [
    'QC Report — California Solar Grid Analysis',
    '=' * 50,
    f'DGStats clean rows: {len(dgstats_clean):,}',
    f'DGStats size outliers: {dgstats_clean["size_outlier"].sum()}',
    f'CAISO monthly rows: {len(caiso):,}',
    f'CAISO net_load_ratio range: [{caiso["net_load_ratio"].min():.3f}, {caiso["net_load_ratio"].max():.3f}]',
]

report_path = PROC / 'qc_report.txt'
report_path.write_text('\n'.join(qc_lines))
print('QC report written to', report_path)

QC report written to /Users/adriannie/Desktop/solar project/solar_grid_analysis/data/processed/qc_report.txt
